# Leçon 18 : Sécuriser les agents IA avec des reçus cryptographiques

## Carnet pratique

Ce carnet vous guide à travers quatre exercices :

1. **Signez votre premier reçu** pour un appel d'outil agent et vérifiez-le.
2. **Altérez le reçu** et observez l’échec de la vérification.
3. **Construisez une chaîne de trois reçus** et confirmez l'intégrité de la chaîne.
4. **Encapsulez un appel d'outil Microsoft Agent Framework** afin que chaque action émette un reçu.

Toutes les primitives cryptographiques sont importées de bibliothèques bien entretenues (`pynacl` pour Ed25519, `jcs` pour le JSON canonique RFC 8785, `hashlib` de la bibliothèque standard Python pour SHA-256). La logique du reçu elle-même est en Python simple que vous pouvez lire et modifier.

Exécutez les cellules dans l'ordre. Chaque section est courte et autonome.


## Configuration

Installez les deux dépendances. Les deux possèdent des licences permissives (Apache-2.0 / MIT).


In [ ]:
!pip install -q pynacl jcs

In [ ]:
import json
import hashlib
import base64
from datetime import datetime, timezone

from nacl import signing
from nacl.exceptions import BadSignatureError
from jcs import canonicalize

## Utilitaires d'aide

Ces deux utilitaires gèrent l'encodage base64url (sans remplissage) et le hachage SHA-256 d'objets arbitraires. Ils permettent au reste du carnet de se concentrer sur la logique propre au reçu.


In [ ]:
def b64url_nopad(data: bytes) -> str:
    """Base64url-encode bytes without padding (RFC 4648 Section 5)."""
    return base64.urlsafe_b64encode(data).decode("ascii").rstrip("=")

def b64url_decode(s: str) -> bytes:
    """Decode a base64url string that may be missing padding."""
    padding = "=" * ((4 - len(s) % 4) % 4)
    return base64.urlsafe_b64decode(s + padding)

def sha256_canonical(obj) -> str:
    """
    SHA-256 hash of a Python object, computed over its JCS-canonical JSON form.
    Returns a 'sha256:' prefixed hex digest so callers can identify the algorithm.
    """
    canonical = canonicalize(obj)
    digest = hashlib.sha256(canonical).hexdigest()
    return f"sha256:{digest}"

## Section 1 : Signez votre premier reçu

Imaginez que notre agent pour **Contoso Travel** vient de rechercher des vols de Sydney à Los Angeles pour un client. Nous voulons enregistrer cet appel d'outil sous forme de reçu signé afin qu'un auditeur futur puisse le vérifier sans avoir à nous faire confiance.

### Étape 1.1 : Générer une clé de signature

En production, la clé de signature de l'agent serait stockée dans un module de sécurité matériel (HSM), Azure Key Vault, ou un magasin sécurisé similaire. Pour cette leçon, nous générons une nouvelle clé en mémoire.


In [ ]:
signing_key = signing.SigningKey.generate()
verify_key = signing_key.verify_key

public_key_b64 = b64url_nopad(bytes(verify_key))
print(f"Public key (Ed25519, 32 bytes): {public_key_b64}")

### Étape 1.2 : Construire la charge utile du reçu

La charge utile contient tout ce que nous voulons que le reçu atteste : qui a agi, quel outil, avec quels arguments, ce qui est revenu, sous quelle politique, et quand. Nous hachons les arguments et le résultat plutôt que de les inclure en ligne afin que le reçu ne divulgue pas de contenu sensible.


In [ ]:
tool_args = {
    "origin": "SYD",
    "destination": "LAX",
    "departure_date": "2026-06-15",
    "passengers": 2,
}

tool_result = [
    {"flight": "QF11", "price": 1850, "stops": 0},
    {"flight": "UA864", "price": 1620, "stops": 1},
    {"flight": "DL11", "price": 1740, "stops": 0},
]

payload = {
    "type": "agent.tool_call.v1",
    "agent_id": "contoso-travel-bot",
    "tool_name": "lookup_flights",
    "tool_args_hash": sha256_canonical(tool_args),
    "result_hash": sha256_canonical(tool_result),
    "policy_id": "contoso-travel-policy-v3",
    "timestamp": datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ"),
    "sequence": 0,
    "previous_receipt_hash": None,
}

print(json.dumps(payload, indent=2))

### Étape 1.3 : Signer et assembler le reçu

Trois étapes :

1. Canonicaliser la charge utile en utilisant JCS afin que deux implémentations produisant le même reçu logique génèrent des octets identiques.
2. Hacher les octets canoniques avec SHA-256.
3. Signer le hachage avec la clé privée Ed25519.

La signature est ensuite attachée à la charge utile originale pour produire le reçu final.


In [ ]:
def sign_receipt(payload: dict, signing_key: signing.SigningKey, verify_key) -> dict:
    """
    Sign a receipt payload. Returns the receipt with attached signature and public key.
    The 'signature' and 'public_key' fields are NOT part of the canonical signed bytes.
    """
    canonical = canonicalize(payload)
    message_hash = hashlib.sha256(canonical).digest()
    signature_bytes = signing_key.sign(message_hash).signature
    return {
        **payload,
        "signature": {
            "alg": "EdDSA",
            "sig": b64url_nopad(signature_bytes),
            "public_key": b64url_nopad(bytes(verify_key)),
        },
    }

receipt = sign_receipt(payload, signing_key, verify_key)
print(json.dumps(receipt, indent=2))

### Étape 1.4 : Vérifier le reçu

La vérification inverse le processus. Nous retirons la signature, recalculons le hachage canonique, et vérifions la signature avec la clé publique contenue dans le reçu.

Un auditeur effectuant cette vérification n’a besoin de rien de notre part à part le reçu lui-même. Aucun service à appeler, aucun annuaire de clés à interroger, aucune confiance requise.


In [ ]:
def verify_receipt(receipt: dict) -> bool:
    """
    Verify a receipt's Ed25519 signature.
    Returns True if valid, False otherwise.
    """
    sig_obj = receipt.get("signature")
    if not sig_obj or sig_obj.get("alg") != "EdDSA":
        return False

    # Reconstruct the payload that was actually signed (everything except signature).
    payload = {k: v for k, v in receipt.items() if k != "signature"}

    canonical = canonicalize(payload)
    message_hash = hashlib.sha256(canonical).digest()

    try:
        verify_key = signing.VerifyKey(b64url_decode(sig_obj["public_key"]))
        verify_key.verify(message_hash, b64url_decode(sig_obj["sig"]))
        return True
    except BadSignatureError:
        return False
    except Exception as exc:
        print(f"Verification error: {exc}")
        return False

is_valid = verify_receipt(receipt)
print(f"Receipt is valid: {is_valid}")

Vous devriez voir `Receipt is valid: True`. L'agent a produit son premier enregistrement d'audit signé cryptographiquement.


## Section 2 : Altérer le reçu

Le but des reçus est qu'ils soient infalsifiables. Prouvons-le.

Nous allons modifier exactement un caractère du reçu et observer l'échec de la vérification.


In [ ]:
import copy

tampered = copy.deepcopy(receipt)

# Modify the policy_id field (this is what an attacker might do to claim
# the action was governed by a more permissive policy than was actually used).
original_policy = tampered["policy_id"]
tampered["policy_id"] = "contoso-travel-policy-PERMISSIVE"

print(f"Original policy_id:  {original_policy}")
print(f"Tampered policy_id:  {tampered['policy_id']}")
print()
print(f"Tampered receipt valid? {verify_receipt(tampered)}")

### Que vient-il de se passer ?

Lorsque nous avons modifié `policy_id`, les octets canoniques ont changé. Le hachage SHA-256 de ces octets a changé. La signature (qui portait sur le hachage original) ne correspond plus au nouveau hachage. La vérification retourne correctement `False`.

Il est impossible de modifier un champ quelconque du reçu et qu'il soit quand même validé, sauf si l'attaquant possède la clé privée. Tant que la clé privée est stockée dans un coffre à clés et que la clé publique est publiée, il est impossible de masquer une altération.

Essayez vous-même : modifiez `tool_name` ou `agent_id` ou `timestamp` dans la cellule ci-dessus et relancez. Chaque modification produit un reçu invalide.


## Section 3 : Chaîner les reçus ensemble

Un seul reçu protège une action. La plupart des agents effectuent de nombreuses actions. Pour rendre toute la séquence détectable en cas de manipulation, nous relions chaque reçu au précédent en incluant le hachage du reçu précédent dans la charge utile du nouveau reçu.

```text
Receipt 0  -->  Receipt 1  -->  Receipt 2
                  |                 |
                  +-- previous_receipt_hash field --+
```

Si quelqu'un supprime ou réorganise un reçu, la chaîne se brise exactement à ce point. La vérification de tout reçu ultérieur échoue car son `previous_receipt_hash` ne correspond plus au hachage réel de son prédécesseur.


In [ ]:
def receipt_hash(receipt: dict) -> str:
    """
    Compute the chain hash of a complete receipt (including signature).
    This becomes the previous_receipt_hash of the next receipt in the chain.
    """
    canonical = canonicalize(receipt)
    digest = hashlib.sha256(canonical).hexdigest()
    return f"sha256:{digest}"

def make_receipt(
    tool_name: str,
    tool_args: dict,
    tool_result,
    sequence: int,
    previous_receipt_hash,
    signing_key,
    verify_key,
    agent_id: str = "contoso-travel-bot",
    policy_id: str = "contoso-travel-policy-v3",
) -> dict:
    """Convenience: build, sign, and return a receipt for one tool call."""
    payload = {
        "type": "agent.tool_call.v1",
        "agent_id": agent_id,
        "tool_name": tool_name,
        "tool_args_hash": sha256_canonical(tool_args),
        "result_hash": sha256_canonical(tool_result),
        "policy_id": policy_id,
        "timestamp": datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ"),
        "sequence": sequence,
        "previous_receipt_hash": previous_receipt_hash,
    }
    return sign_receipt(payload, signing_key, verify_key)

In [ ]:
# Build a chain of three receipts: search, hold, book.
r0 = make_receipt(
    tool_name="lookup_flights",
    tool_args={"origin": "SYD", "destination": "LAX", "date": "2026-06-15"},
    tool_result=[{"flight": "QF11", "price": 1850}],
    sequence=0,
    previous_receipt_hash=None,
    signing_key=signing_key,
    verify_key=verify_key,
)

r1 = make_receipt(
    tool_name="hold_seat",
    tool_args={"flight": "QF11", "seat": "14A", "hold_minutes": 30},
    tool_result={"hold_id": "H8472", "expires_at": "2026-06-15T15:00:00Z"},
    sequence=1,
    previous_receipt_hash=receipt_hash(r0),
    signing_key=signing_key,
    verify_key=verify_key,
)

r2 = make_receipt(
    tool_name="confirm_booking",
    tool_args={"hold_id": "H8472", "payment_token": "tok_redacted"},
    tool_result={"booking_ref": "CT-09182", "status": "confirmed"},
    sequence=2,
    previous_receipt_hash=receipt_hash(r1),
    signing_key=signing_key,
    verify_key=verify_key,
)

chain = [r0, r1, r2]
for i, r in enumerate(chain):
    print(f"Receipt {i}: tool={r['tool_name']}, prev={r['previous_receipt_hash']}")

In [ ]:
def verify_chain(chain: list) -> list[dict]:
    """
    Verify a sequence of receipts:
      1. Each receipt's signature must verify.
      2. Each receipt (except the genesis) must reference the previous receipt's hash.
      3. Sequence numbers must match each receipt's zero-based position in the chain.
    Returns a list of per-receipt result dicts.
    """
    results = []
    for i, receipt in enumerate(chain):
        sig_ok = verify_receipt(receipt)

        if i == 0:
            chain_ok = receipt["previous_receipt_hash"] is None
        else:
            expected = receipt_hash(chain[i - 1])
            chain_ok = receipt["previous_receipt_hash"] == expected

        seq_ok = receipt["sequence"] == i

        results.append({
            "index": i,
            "tool": receipt["tool_name"],
            "signature_valid": sig_ok,
            "chain_link_valid": chain_ok,
            "sequence_valid": seq_ok,
            "overall_valid": sig_ok and chain_ok and seq_ok,
        })
    return results

for r in verify_chain(chain):
    status = "VALID" if r["overall_valid"] else "INVALID"
    print(f"Receipt {r['index']} ({r['tool']:>18}): {status}")

Brisez maintenant la chaîne en altérant le reçu du milieu, puis vérifiez de nouveau. Le reçu altéré échoue à la vérification de sa signature, ET le reçu suivant échoue à la vérification du lien de chaîne (car son `previous_receipt_hash` ne correspond plus au hachage du reçu modifié du milieu).


In [ ]:
# Tamper with the middle receipt: change the hold duration to something
# more permissive than was actually authorized.
tampered_chain = [copy.deepcopy(r) for r in chain]
tampered_chain[1]["tool_args_hash"] = sha256_canonical(
    {"flight": "QF11", "seat": "14A", "hold_minutes": 9999}
)

for r in verify_chain(tampered_chain):
    status = "VALID" if r["overall_valid"] else "INVALID"
    why = ""
    if not r["overall_valid"]:
        reasons = []
        if not r["signature_valid"]:
            reasons.append("signature")
        if not r["chain_link_valid"]:
            reasons.append("chain link")
        if not r["sequence_valid"]:
            reasons.append("sequence")
        why = " (failed: " + ", ".join(reasons) + ")"
    print(f"Receipt {r['index']} ({r['tool']:>18}): {status}{why}")

Le reçu 0 est toujours valide (il n’a pas été modifié et n’a aucun prédécesseur dont il dépend). Le reçu 1 échoue à sa vérification de signature car nous avons changé `tool_args_hash`. Le reçu 2 échoue à sa vérification de chaîne car son `previous_receipt_hash` a été calculé à partir du reçu 1 original (maintenant modifié).

Même si un attaquant re-signe le reçu 1 modifié (ce qu’il ne peut pas faire sans la clé privée), l’incohérence de chaîne dans le reçu 2 révélerait toujours la falsification. Pour masquer la modification, l’attaquant devrait re-signer chaque reçu à partir du point de modification, ce qui nécessite la possession de la clé privée.


## Section 4 : Encapsuler un appel d’outil agent avec signature de reçu

Dans un déploiement réel, vous ne voulez pas que chaque auteur d’agent se souvienne d’appeler `make_receipt`. Vous souhaitez que la signature des reçus soit automatique pour chaque invocation d’outil.

Voici le modèle le plus simple : une classe wrapper qui prend n’importe quelle fonction d’outil callable et retourne une version émettant un reçu. Cela s’adapte à n’importe quel framework d’agent, y compris le Microsoft Agent Framework (`agent_framework.azure`).

Si vous n’avez pas de projet Azure AI Foundry configuré, la maquette locale ci-dessous démontre quand même le modèle.


In [ ]:
class ReceiptedTool:
    """
    Wraps a tool function so every invocation produces a signed receipt.
    Receipts are appended to a chain held by this object.

    Accepts both positional and keyword arguments. The receipt's
    tool_args field records args (as a list) and kwargs (as a dict)
    so the canonical hash binds to whichever the caller supplied.
    """

    def __init__(self, name: str, fn, signing_key, verify_key, agent_id: str, policy_id: str):
        self.name = name
        self.fn = fn
        self.signing_key = signing_key
        self.verify_key = verify_key
        self.agent_id = agent_id
        self.policy_id = policy_id
        self.receipts: list = []

    def __call__(self, *args, **kwargs):
        result = self.fn(*args, **kwargs)
        previous_hash = receipt_hash(self.receipts[-1]) if self.receipts else None
        receipt = make_receipt(
            tool_name=self.name,
            tool_args={"args": list(args), "kwargs": kwargs},
            tool_result=result,
            sequence=len(self.receipts),
            previous_receipt_hash=previous_hash,
            signing_key=self.signing_key,
            verify_key=self.verify_key,
            agent_id=self.agent_id,
            policy_id=self.policy_id,
        )
        self.receipts.append(receipt)
        return result

In [ ]:
# Example tool: a mock flight lookup. In a real Microsoft Agent Framework deployment,
# this would be a function passed to AzureAIProjectAgentProvider as a tool.
def mock_lookup_flights(origin: str, destination: str, departure_date: str) -> list:
    return [
        {"flight": "QF11", "price": 1850, "stops": 0},
        {"flight": "UA864", "price": 1620, "stops": 1},
    ]

# Wrap it with receipt signing.
receipted_lookup = ReceiptedTool(
    name="lookup_flights",
    fn=mock_lookup_flights,
    signing_key=signing_key,
    verify_key=verify_key,
    agent_id="contoso-travel-bot",
    policy_id="contoso-travel-policy-v3",
)

# Use the wrapped tool exactly like the original.
results_a = receipted_lookup(origin="SYD", destination="LAX", departure_date="2026-06-15")
results_b = receipted_lookup(origin="SYD", destination="NRT", departure_date="2026-07-02")
results_c = receipted_lookup(origin="MEL", destination="SIN", departure_date="2026-08-10")

print(f"Tool was called {len(receipted_lookup.receipts)} times.")
print(f"Each call produced a signed receipt linked to the previous one.")
print()

for r in verify_chain(receipted_lookup.receipts):
    status = "VALID" if r["overall_valid"] else "INVALID"
    print(f"Receipt {r['index']} ({r['tool']}): {status}")

### Intégration avec le Microsoft Agent Framework

Le wrapper `ReceiptedTool` ci-dessus est indépendant du framework. Pour l'utiliser dans un agent construit avec le Microsoft Agent Framework, enregistrez la fonction enveloppée comme un outil. Un croquis (vous remplaceriez la simulation par un véritable enregistrement d'outil Azure AI Foundry) :

```python
# Pseudocode montrant la forme de l'intégration.
# from agent_framework.azure import AzureAIProjectAgentProvider
# from azure.identity import AzureCliCredential
#
# provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())
# agent = provider.create_agent(
#     instructions="Vous êtes un agent de Contoso Travel ...",
#     tools=[receipted_lookup],   # l'outil encapsulé, pas la fonction brute
# )
# response = agent.run("Trouve des vols de Sydney à Los Angeles en juin.")
#
# # Après l'exécution, chaque appel d'outil effectué par l'agent a un reçu signé :
# audit_chain = receipted_lookup.receipts
```

Le framework de l'agent n'a pas besoin de connaître quoi que ce soit sur les reçus. La signature des reçus est enveloppée autour de l'outil, pas intégrée dans le framework. C'est ainsi que vous ajoutez de la provenance au code existant de l'agent sans réécrire l'agent.


## Récapitulatif et défi supplémentaire

Vous avez :

- Généré une paire de clés Ed25519.  
- Construit et signé un reçu pour un appel d'outil agent.  
- Vérifié le reçu hors ligne en utilisant uniquement la clé publique.  
- Altéré un reçu et observé l’échec de la vérification.  
- Construit une séquence chaînée par hachage de trois reçus.  
- Altéré le milieu de la chaîne et observé à la fois l’échec de la signature et l’échec de la chaîne.  
- Enveloppé une fonction d’outil avec une signature automatique des reçus.

**Défi supplémentaire.** Étendez le schéma du reçu avec un champ `request_id` (un UUID pour le traçage distribué). Mettez à jour `make_receipt` pour l’inclure, et confirmez que les reçus se vérifient toujours de bout en bout. Puis modifiez ce champ après signature et confirmez que la vérification échoue. Cela vous oblige à intégrer comment chaque octet du codage canonique contribue à la signature.

**Limite importante.** Les reçus prouvent trois choses et seulement trois choses : l’attribution (cette clé a signé ce contenu), l’intégrité (le contenu n’a pas changé depuis la signature) et l’ordre (ce reçu est venu après ce reçu-là). Ils ne prouvent PAS que l’action de l’agent était correcte, que la politique nommée dans `policy_id` a réellement été évaluée, ou que l’agent a suivi toutes les règles. Les reçus sont une fondation. La gouvernance est le système que vous construisez par-dessus.

Relisez la README de la leçon avec cette limite en tête. L’erreur la plus courante des équipes avec les reçus est de supposer que « nous avons des reçus » signifie « nous sommes gouvernés ». Ce n’est pas le cas. Les reçus rendent le comportement des agents auditable. Ils ne le rendent pas correct.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Avertissement** :
Ce document a été traduit à l'aide du service de traduction automatique [Co-op Translator](https://github.com/Azure/co-op-translator). Bien que nous nous efforçions d'assurer l'exactitude, veuillez noter que les traductions automatisées peuvent contenir des erreurs ou des inexactitudes. Le document original dans sa langue native doit être considéré comme la source faisant autorité. Pour les informations critiques, il est recommandé de recourir à une traduction professionnelle réalisée par un humain. Nous ne saurions être tenus responsables des malentendus ou erreurs d'interprétation découlant de l'utilisation de cette traduction.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
